# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show the available record sets
print('Available Record Sets:')
for rset in meta.record_sets:
    print(f"  @id: {rset.id}  | name: {rset.name if hasattr(rset,'name') else ''}")

# List fields and columns for each record set
for rset in meta.record_sets:
    print(f"\nRecord Set {rset.id}")
    print('  Fields:')
    for fld in rset.fields:
        print(f"    @id: {fld.id}  | name: {fld.name if hasattr(fld,'name') else ''}  | dataType: {fld.data_type if hasattr(fld,'data_type') else ''}")
    print('  Columns in source files:')
    for fobj in getattr(rset, 'file_objects', []):
        for col in getattr(fobj, 'columns', []):
            print(f"    @id: {col.id}  | name: {col.name if hasattr(col,'name') else ''}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Identify all record set @ids
record_set_ids = [rset.id for rset in meta.record_sets]
print(f"Will extract from record sets: {record_set_ids}")

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"\nColumns in record set {rec_id}:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** All references to fields and columns should use their `@id` values as extracted above. Adjust the field IDs as necessary for your dataset.

In [ ]:
# Choose the first record set for demonstration
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id].copy()
    print(f"Analyzing record set: {rs_id}\n")
else:
    print("No record sets available.")

# List numeric columns for reference
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields:", numeric_fields)

# Pick the first numeric field if present
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise RuntimeError("No numeric fields available in this record set.")

# Example: Filter on this numeric field (arbitrary threshold)
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize filtered field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely categorical field
categorical_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
group_field = categorical_fields[0] if categorical_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
if numeric_fields:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in record set {rs_id}")
    plt.tight_layout()
    plt.show()
    
    # Boxplot by group if group field exists
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR² dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using `mlcroissant`. We demonstrated how to programmatically extract and analyze records from the dataset by referencing each entity with its unique `@id`, filtered and normalized numeric data, and visualized data distributions. 

Refer to the field and record set `@id`s listed in Section 2 to programmatically extend this EDA, or apply similar approaches to other record sets defined in the schema.